# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process the [FAIR²](https://sen.science/doi/10.71728/senscience.y7m0-f273) dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant-python) library. All dataset entities—record sets, fields, and columns—are referenced using their `@id` as per Croissant best practices.

### Dataset Source
Croissant schema URL:
`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Install mlcroissant if not present
!pip install -q mlcroissant

## 1. Data Loading
Load the metadata and get a sense of the dataset using `mlcroissant`. All further references to record sets, fields, and columns are done via their `@id`s.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
from pprint import pprint

# Define the dataset Croissant JSON-LD schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load dataset: this fetches and parses all Croissant-compliant metadata and structure
dataset = mlc.Dataset(croissant_url)

# Print a summary of the dataset using metadata object attributes
print(f"Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}\n")
if hasattr(dataset.metadata, 'keywords'):
    print(f"Keywords: {dataset.metadata.keywords}\n")
print(f"License: {dataset.metadata.license}")
print(f"Version: {dataset.metadata.version}")
print(f"Coverage: {dataset.metadata.spatialCoverage}")

## 2. Data Overview
List available record sets in the dataset, with their `@id` and available fields (also by `@id`). This helps identify what data is present and how it is structured.

In [ ]:
# List all record sets and their fields by @id. This ensures precise, stable referencing.

print("\u25B6 Available Record Sets:")
record_set_ids = []
for rs in dataset.record_sets:
    print(f"  - @id: {rs.id} ; name: {rs.name}")
    record_set_ids.append(rs.id)
    print("    Available fields:")
    for f in rs.fields:
        print(f"      * @id: {f.id} ; name: {f.name} ; dataType: {getattr(f, 'data_type', '<unknown>')}")
    print("")
if not record_set_ids:
    print("  No record sets were found in this dataset. \nPerhaps the distribution points to external files.\n")

## 3. Data Extraction
Extract data for each detected record set into a pandas DataFrame, referencing each entity by its `@id`. Even if only one record set is present, this structure can handle multiple ones. The next cell loads all of them into named variables for convenience.

In [ ]:
# Extract and display a sample of each record set
# If no record sets found, attempt to load via distribution objects.

# Use dictionary {record_set_id: DataFrame}
dataframes = dict()
if record_set_ids:
    for rs_id in record_set_ids:
        # Load all records from the record set
        records_iter = dataset.records(record_set=rs_id)
        records = list(records_iter)
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Record set {rs_id} loaded: {df.shape[0]} rows, {df.shape[1]} columns.")
    # Show columns of first record set as a sample
    display_id = record_set_ids[0]
    print(f"\nColumns in first record set (@id={display_id}):")
    print(dataframes[display_id].columns.tolist())
    print("\nSample rows:")
    display_rows = 5 if dataframes[display_id].shape[0] > 5 else dataframes[display_id].shape[0]
    print(dataframes[display_id].head(display_rows))
else:
    print("No record sets found in this dataset (possibly only metadata present or custom structure). If you expect tabular data, check the distribution URLs:")
    if hasattr(dataset.metadata, 'distribution'):
        for dist in dataset.metadata.distribution:
            print(f"  - {getattr(dist, 'id', dist)}")

## 4. Exploratory Data Analysis (EDA)
In this section, common data prep operations are shown: filtering numeric columns, normalizing, grouping—**using only `@id`s** for all columns/fields. If no record sets with fields/columns are present, this cell will indicate as such.

In [ ]:
# Example EDA: Select the first tabular record set and pick a numeric field
import numpy as np

if dataframes:
    # Pick the first record set for demonstration
    target_rs_id = record_set_ids[0]
    df = dataframes[target_rs_id]
    print(f"Working with record set: {target_rs_id} ({df.shape[0]} rows)")

    # Display columns to choose a numeric one by @id
    print("Columns/fields (@id):", df.columns.tolist())

    # Automatically pick a likely-numeric field by inspecting dtypes
    likely_numeric = [col for col in df.select_dtypes(include=[np.number]).columns]
    if likely_numeric:
        numeric_field_id = likely_numeric[0]
        print(f"Using numeric field: {numeric_field_id}")

        # Basic filtering: select records with a value above the mean
        threshold = df[numeric_field_id].mean()
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"\nFiltered records with {numeric_field_id} > mean ({threshold:.2f}): {filtered_df.shape[0]} rows")
        print(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / (filtered_df[numeric_field_id].std() + 1e-12)
        print(f"\nNormalized {numeric_field_id} (z-score):")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try grouping by another field (prefer non-numeric for categorical grouping)
        non_numeric_cols = [c for c in df.columns if c != numeric_field_id and df[c].dtype == object]
        group_field_id = non_numeric_cols[0] if non_numeric_cols else None
        if group_field_id:
            grouped = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"\nGrouped by {group_field_id} (means):")
            print(grouped.head())
        else:
            print("No suitable categorical field (@id) found for grouping.")
    else:
        print("No numeric fields found in the selected record set.")
else:
    print("No record sets/dataframes available for EDA.")

## 5. Visualization
Plot distributions or relationships between fields. If numeric fields are available, their distributions and relationships are visualized using their `@id`.

In [ ]:
import matplotlib.pyplot as plt

if dataframes and 'numeric_field_id' in locals():
    plt.figure(figsize=(6,4))
    plt.hist(df[numeric_field_id].dropna(), bins=30, color='skyblue')
    plt.title(f"Distribution of field (@id): {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If grouping field present, show boxplot per group
    if 'group_field_id' in locals() and group_field_id is not None:
        plt.figure(figsize=(9,5))
        df.boxplot(column=numeric_field_id, by=group_field_id, grid=False, rot=30)
        plt.title(f"{numeric_field_id} by {group_field_id} (all data)")
        plt.suptitle("")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("Visualization not possible: No numeric or grouped field detected.")

## 6. Conclusion
This notebook showed how to:
- Load Croissant datasets using their schema URL
- Explore structure via record set and field `@id`s
- Load record data directly into DataFrames, referencing columns by canonical `@id`
- Apply EDA routines: filtering, normalization, grouping
- Visualize numeric distributions and relationships

This approach ensures interoperability, reproducibility, and makes it easy to reference specific dataset components for downstream tasks, following the FAIR principles.